<a href="https://colab.research.google.com/github/StathisDevves/Industrial/blob/main/MiniMill%207%20Days%20Lowest%20Prices%20set%202025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

# =========================================================
# LOWEST 8-DAY MINI-MILL OPTIMIZATION
# =========================================================
# INPUT FILE:
#   Electricity_Prices_2025_filtered_8760.xlsx
#
# LOGIC:
#   1. Read the full-year hourly price file
#   2. Find the 8 consecutive days (192 hours) with the lowest average price
#   3. For each of these 8 days independently:
#        - optimize the start hour of EAF1.1
#        - minimize daily total cost over the fixed 24-process sequence
#   4. Create a global process time t = 1 ... 192
#   5. Export the Excel workbook:
#        - Summary
#        - 8-Day Window Ranking
#        - Selected Prices
#        - Daily Optimization
#        - Optimized Schedule
#        - Notes
# =========================================================

input_file = "Electricity_Prices_2025_filtered_8760.xlsx"
output_file = "Lowest_8Day_Mini_Mill_Optimized_Schedule.xlsx"

# ---------------------------------------------------------
# 1. READ INPUT
# ---------------------------------------------------------
raw = pd.read_excel(input_file)

if raw.empty:
    raise ValueError("Input file is empty.")

raw.columns = [str(c).strip() for c in raw.columns]

# ---------------------------------------------------------
# 2. DETECT DATETIME COLUMN
# ---------------------------------------------------------
datetime_col = None
for col in raw.columns:
    col_lower = col.lower()
    if any(k in col_lower for k in ["datetime", "date", "time", "timestamp"]):
        parsed = pd.to_datetime(raw[col], errors="coerce")
        if parsed.notna().sum() > 0:
            datetime_col = col
            raw[col] = parsed
            break

if datetime_col is None:
    parsed = pd.to_datetime(raw.iloc[:, 0], errors="coerce")
    if parsed.notna().sum() > 0:
        datetime_col = raw.columns[0]
        raw[datetime_col] = parsed
    else:
        raise ValueError("Could not detect a datetime column.")

# ---------------------------------------------------------
# 3. DETECT PRICE COLUMN
# ---------------------------------------------------------
price_col = None
for col in raw.columns:
    if col == datetime_col:
        continue
    if "price" in col.lower():
        price_col = col
        break

if price_col is None:
    for col in raw.columns:
        if col == datetime_col:
            continue
        if pd.api.types.is_numeric_dtype(raw[col]):
            price_col = col
            break

if price_col is None:
    raise ValueError("Could not detect a price column.")

df = raw[[datetime_col, price_col]].copy()
df.columns = ["Datetime", "Price"]
df = df.dropna(subset=["Datetime", "Price"]).copy()
df["Datetime"] = pd.to_datetime(df["Datetime"])
df["Price"] = pd.to_numeric(df["Price"], errors="coerce")
df = df.dropna(subset=["Price"]).sort_values("Datetime").reset_index(drop=True)

# Add date and hour
df["Date"] = df["Datetime"].dt.date
df["Hour"] = df["Datetime"].dt.hour

# Keep only complete 24-hour days
day_counts = df.groupby("Date").size()
complete_days = day_counts[day_counts == 24].index
df = df[df["Date"].isin(complete_days)].copy().reset_index(drop=True)

if df.empty:
    raise ValueError("No complete 24-hour days found in the dataset.")

# ---------------------------------------------------------
# 4. FIND LOWEST-AVERAGE 8 CONSECUTIVE DAYS
# ---------------------------------------------------------
daily_avg = (
    df.groupby("Date", as_index=False)["Price"]
    .mean()
    .rename(columns={"Price": "Daily Average Price"})
)

daily_avg["Date"] = pd.to_datetime(daily_avg["Date"])
daily_avg = daily_avg.sort_values("Date").reset_index(drop=True)

window_rows = []
best_window_start_idx = None
best_window_avg = None

for i in range(len(daily_avg) - 8 + 1):
    window = daily_avg.iloc[i:i+8].copy()
    start_date = window["Date"].iloc[0]
    end_date = window["Date"].iloc[-1]
    avg_price = window["Daily Average Price"].mean()

    # Check consecutive dates
    expected = pd.date_range(start=start_date, periods=8, freq="D")
    if not (window["Date"].tolist() == list(expected)):
        continue

    window_rows.append({
        "Window Start": start_date,
        "Window End": end_date,
        "Average Price Over 8 Days": avg_price
    })

    if best_window_avg is None or avg_price < best_window_avg:
        best_window_avg = avg_price
        best_window_start_idx = i

if best_window_start_idx is None:
    raise ValueError("Could not find any valid 8 consecutive complete days.")

df_window_ranking = pd.DataFrame(window_rows).sort_values(
    "Average Price Over 8 Days"
).reset_index(drop=True)

best_start_date = daily_avg.iloc[best_window_start_idx]["Date"]
best_end_date = best_start_date + pd.Timedelta(days=7)

selected = df[
    (pd.to_datetime(df["Date"]) >= best_start_date) &
    (pd.to_datetime(df["Date"]) <= best_end_date)
].copy()

if len(selected) != 8 * 24:
    raise ValueError(f"Expected 192 hourly rows for selected 8-day window, found {len(selected)}.")

selected = selected.sort_values("Datetime").reset_index(drop=True)

# ---------------------------------------------------------
# 5. DEFINE FIXED 24-PROCESS SEQUENCE
# ---------------------------------------------------------
stage_sequence = [
    ("Step 1", "EAF1.1", 72, 1),
    ("Step 1", "EAF1.2", 78, 2),
    ("Step 1", "EAF1.3", 80, 3),
    ("Step 1", "EAF1.4", 72, 4),

    ("Step 2", "Secondary Downstream 1.1", 32, 5),
    ("Step 2", "Secondary Downstream 1.2", 26, 6),
    ("Step 2", "Secondary Downstream 1.3", 20, 7),

    ("Step 3", "SD sequence Blue Colour", 15, 8),

    ("Step 4", "Minimum Critical Load 1", 12, 9),
    ("Step 4", "Minimum Critical Load 2", 12, 10),
    ("Step 4", "Minimum Critical Load 3", 12, 11),
    ("Step 4", "Minimum Critical Load 4", 12, 12),
    ("Step 4", "Minimum Critical Load 5", 12, 13),
    ("Step 4", "Minimum Critical Load 6", 12, 14),

    ("Step 5", "Preparation Blue 1", 15, 15),
    ("Step 5", "Preparation Blue 2", 24, 16),
    ("Step 5", "Preparation Blue 3", 38, 17),

    ("Step 6", "EAF2.1", 72, 18),
    ("Step 6", "EAF2.2", 78, 19),
    ("Step 6", "EAF2.3", 80, 20),
    ("Step 6", "EAF2.4", 76, 21),

    ("Step 7", "Secondary Downstream 2.1", 28, 22),
    ("Step 7", "Secondary Downstream 2.2", 24, 23),
    ("Step 7", "Secondary Downstream 2.3", 20, 24),
]

# ---------------------------------------------------------
# 6. DAILY SCHEDULE BUILDER
# ---------------------------------------------------------
def build_day_schedule(day_df, start_hour):
    """
    One daily 24-hour circular optimization.
    """
    day_df = day_df.sort_values("Hour").reset_index(drop=True)
    if len(day_df) != 24:
        raise ValueError("Each daily optimization requires exactly 24 hourly rows.")

    prices = day_df["Price"].tolist()
    datetimes = day_df["Datetime"].tolist()
    hours = day_df["Hour"].tolist()

    rows = []
    total_cost = 0.0

    for i, (step, phase, load, t_in_day) in enumerate(stage_sequence):
        idx = (start_hour + i) % 24
        dt = datetimes[idx]
        hr = hours[idx]
        price = prices[idx]
        cost = load * price
        total_cost += cost

        rows.append({
            "t_in_day": t_in_day,
            "Step": step,
            "Production Phase": phase,
            "Total Load (MWh)": load,
            "Assigned Hour": hr,
            "Assigned Datetime": dt,
            "Price": price,
            "Cost = Load x Price": cost
        })

    return pd.DataFrame(rows), total_cost

# ---------------------------------------------------------
# 7. OPTIMIZE EACH OF THE 8 DAYS
# ---------------------------------------------------------
daily_optimization_rows = []
optimized_schedule_rows = []

selected_dates = sorted(selected["Date"].unique())
global_t = 1
total_8day_cost = 0.0

for d in selected_dates:
    day_df = selected[selected["Date"] == d].copy()

    if len(day_df) != 24:
        raise ValueError(f"Date {d} does not contain 24 hourly rows.")

    best_start_hour = None
    best_daily_cost = None
    best_schedule = None

    for start_hour in range(24):
        schedule_df, total_cost = build_day_schedule(day_df, start_hour)

        if best_daily_cost is None or total_cost < best_daily_cost:
            best_daily_cost = total_cost
            best_start_hour = start_hour
            best_schedule = schedule_df.copy()

    total_8day_cost += best_daily_cost

    daily_optimization_rows.append({
        "Date": pd.Timestamp(d),
        "Optimal Start Hour (EAF1.1)": best_start_hour,
        "Minimum Daily Cost": best_daily_cost
    })

    for _, row in best_schedule.iterrows():
        optimized_schedule_rows.append({
            "Global t": global_t,
            "Date": pd.Timestamp(d),
            "t_in_day": int(row["t_in_day"]),
            "Step": row["Step"],
            "Production Phase": row["Production Phase"],
            "Total Load (MWh)": row["Total Load (MWh)"],
            "Assigned Hour": int(row["Assigned Hour"]),
            "Assigned Datetime": row["Assigned Datetime"],
            "Price": row["Price"],
            "Cost = Load x Price": row["Cost = Load x Price"]
        })
        global_t += 1

df_daily_optimization = pd.DataFrame(daily_optimization_rows)
df_optimized_schedule = pd.DataFrame(optimized_schedule_rows)

# ---------------------------------------------------------
# 8. SUMMARY + NOTES
# ---------------------------------------------------------
summary_df = pd.DataFrame({
    "Metric": [
        "Selected 8-day window start",
        "Selected 8-day window end",
        "Lowest 8-day average price (EUR/MWh)",
        "Number of optimized days",
        "Total process times",
        "Total optimized 8-day cost (EUR)"
    ],
    "Value": [
        best_start_date,
        best_end_date,
        best_window_avg,
        8,
        24 * 8,
        total_8day_cost
    ]
})

notes_df = pd.DataFrame({
    "Notes": [
        "The model scans all valid 8 consecutive complete days in the dataset.",
        "It selects the 8-day window with the lowest average electricity price.",
        "Within that selected window, each day is optimized independently as one 24-process mini-mill circle.",
        "Global process time runs from t=1 to t=192.",
        "The objective minimizes the sum of Load x Price for each day, then aggregates the total 8-day cost."
    ]
})

# ---------------------------------------------------------
# 9. WRITE EXCEL
# ---------------------------------------------------------
with pd.ExcelWriter(output_file, engine="openpyxl") as writer:
    summary_df.to_excel(writer, sheet_name="Summary", index=False)
    df_window_ranking.to_excel(writer, sheet_name="8-Day Window Ranking", index=False)
    selected[["Datetime", "Date", "Hour", "Price"]].to_excel(writer, sheet_name="Selected Prices", index=False)
    df_daily_optimization.to_excel(writer, sheet_name="Daily Optimization", index=False)
    df_optimized_schedule.to_excel(writer, sheet_name="Optimized Schedule", index=False)
    notes_df.to_excel(writer, sheet_name="Notes", index=False)

# ---------------------------------------------------------
# 10. FORMAT EXCEL
# ---------------------------------------------------------
wb = load_workbook(output_file)

header_fill = PatternFill("solid", fgColor="1F4E78")
header_font = Font(color="FFFFFF", bold=True)
thin = Side(style="thin", color="BFBFBF")
border = Border(left=thin, right=thin, top=thin, bottom=thin)

phase_fills = {
    "EAF": PatternFill("solid", fgColor="FCE4D6"),
    "Secondary": PatternFill("solid", fgColor="E2F0D9"),
    "SD sequence Blue Colour": PatternFill("solid", fgColor="D9EAF7"),
    "Minimum Critical Load": PatternFill("solid", fgColor="F4CCCC"),
    "Preparation Blue": PatternFill("solid", fgColor="D9E1F2"),
}

for ws in wb.worksheets:
    for cell in ws[1]:
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = border

    for row in ws.iter_rows(min_row=2):
        for cell in row:
            cell.border = border
            cell.alignment = Alignment(vertical="center")

    for col_cells in ws.columns:
        max_len = 0
        col_letter = col_cells[0].column_letter
        for cell in col_cells:
            val = "" if cell.value is None else str(cell.value)
            max_len = max(max_len, len(val))
        ws.column_dimensions[col_letter].width = min(max_len + 2, 36)

# Number formatting
for sheet_name in ["Summary", "8-Day Window Ranking", "Selected Prices", "Daily Optimization", "Optimized Schedule"]:
    ws = wb[sheet_name]
    headers = [c.value for c in ws[1]]
    for col_idx, header in enumerate(headers, start=1):
        if header and ("Price" in str(header) or "Cost" in str(header)):
            for row in range(2, ws.max_row + 1):
                ws.cell(row=row, column=col_idx).number_format = "0.00"

# Datetime formatting
for sheet_name in ["Summary", "8-Day Window Ranking", "Selected Prices", "Daily Optimization", "Optimized Schedule"]:
    ws = wb[sheet_name]
    headers = [c.value for c in ws[1]]
    for col_idx, header in enumerate(headers, start=1):
        if header and ("Date" in str(header) or "Datetime" in str(header) or "window" in str(header).lower()):
            for row in range(2, ws.max_row + 1):
                cell = ws.cell(row=row, column=col_idx)
                if cell.value is not None:
                    try:
                        cell.number_format = "yyyy-mm-dd hh:mm"
                    except Exception:
                        pass

# Phase coloring
ws_opt = wb["Optimized Schedule"]
headers_opt = [c.value for c in ws_opt[1]]
phase_col = headers_opt.index("Production Phase") + 1

for row in range(2, ws_opt.max_row + 1):
    phase = str(ws_opt.cell(row=row, column=phase_col).value)

    fill = None
    if phase.startswith("EAF"):
        fill = phase_fills["EAF"]
    elif phase.startswith("Secondary"):
        fill = phase_fills["Secondary"]
    elif phase == "SD sequence Blue Colour":
        fill = phase_fills["SD sequence Blue Colour"]
    elif phase.startswith("Minimum Critical Load"):
        fill = phase_fills["Minimum Critical Load"]
    elif phase.startswith("Preparation Blue"):
        fill = phase_fills["Preparation Blue"]

    if fill:
        ws_opt.cell(row=row, column=phase_col).fill = fill

wb.save(output_file)

# ---------------------------------------------------------
# 11. PRINT RESULTS
# ---------------------------------------------------------
print(f"Output saved to: {output_file}")
print(f"Selected 8-day window: {best_start_date} to {best_end_date}")
print(f"Lowest 8-day average price: {best_window_avg:.2f} EUR/MWh")
print(f"Total optimized 8-day cost: {total_8day_cost:,.2f} EUR")

print("\nDaily optimal start hours:")
for _, row in df_daily_optimization.iterrows():
    print(f"{row['Date'].date()} -> {int(row['Optimal Start Hour (EAF1.1)'])}")

Output saved to: Lowest_8Day_Mini_Mill_Optimized_Schedule.xlsx
Selected 8-day window: 2025-04-27 00:00:00 to 2025-05-04 00:00:00
Lowest 8-day average price: 54.10 EUR/MWh
Total optimized 8-day cost: 244,191.31 EUR

Daily optimal start hours:
2025-04-27 -> 12
2025-04-28 -> 12
2025-04-29 -> 12
2025-04-30 -> 11
2025-05-01 -> 12
2025-05-02 -> 12
2025-05-03 -> 12
2025-05-04 -> 11
